In [96]:
!pip -q install datasets transformers tokenizers sentencepiece tqdm

In [97]:
from datasets import load_dataset
from tqdm.auto import tqdm
import json
import os

In [98]:
datasets = {
    "openhermes": load_dataset("teknium/OpenHermes-2.5", split="train", streaming=True),
    "openorca": load_dataset("Open-Orca/OpenOrca", split="train", streaming=True),
    "ultrachat": load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True),
    "wildchat": load_dataset("allenai/WildChat", split="train", streaming=True),
    "ultrafeedback": load_dataset("argilla/ultrafeedback-binarized-preferences-cleaned", split="train", streaming=True),
    "magpie": load_dataset("Magpie-Align/Magpie-Pro-DPO-100K-v0.1", split="train", streaming=True),
}

In [99]:
from tokenizers import Tokenizer
import numpy as np
import random

In [100]:
TOKENIZER_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

In [101]:
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")

In [102]:
print(f"Vocabulary Size : {tokenizer.get_vocab_size():,}")
print(f"BOS ID : {BOS_ID}")
print(f"EOS ID : {EOS_ID}")

Vocabulary Size : 45,000
BOS ID : 2
EOS ID : 3


In [103]:
import os
import numpy as np

OUTPUT_DIR = "/kaggle/working/virgo_chat"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_BIN = os.path.join(OUTPUT_DIR, "train.bin")
VAL_BIN = os.path.join(OUTPUT_DIR, "val.bin")

train_file = open(TRAIN_BIN, "wb")
val_file = open(VAL_BIN, "wb")

train_tokens = 0
val_tokens = 0

random.seed(42)

print(OUTPUT_DIR)

/kaggle/working/virgo_chat


In [104]:
def convert_openhermes(sample):
    messages = []

    system = sample.get("system_prompt")
    if system and system.strip():
        messages.append({
            "role": "system",
            "content": system.strip()
        })

    for msg in sample.get("conversations", []):

        role = msg.get("from", "").lower()

        if role == "human":
            role = "user"
        elif role == "gpt":
            role = "assistant"

        if role not in ("system", "user", "assistant"):
            continue

        content = msg.get("value", "").strip()

        if not content:
            continue

        messages.append({
            "role": role,
            "content": content
        })

    return messages

In [105]:
def convert_openorca(sample):
    messages = []

    system = sample.get("system_prompt", "")
    if isinstance(system, str) and system.strip():
        messages.append({
            "role": "system",
            "content": system.strip()
        })

    question = sample.get("question", "")
    if isinstance(question, str) and question.strip():
        messages.append({
            "role": "user",
            "content": question.strip()
        })

    response = sample.get("response", "")
    if isinstance(response, str) and response.strip():
        messages.append({
            "role": "assistant",
            "content": response.strip()
        })

    return messages

In [106]:
def convert_ultrachat(sample):
    messages = []

    for msg in sample.get("messages", []):
        role = msg.get("role", "").lower()

        if role not in {"system", "user", "assistant"}:
            continue

        content = msg.get("content", "").strip()

        if content:
            messages.append({
                "role": role,
                "content": content
            })

    return messages

In [107]:
def convert_wildchat(sample):
    messages = []

    for msg in sample.get("conversation", []):
        role = msg.get("role", "").lower()

        if role not in {"system", "user", "assistant"}:
            continue

        content = msg.get("content", "").strip()

        if content:
            messages.append({
                "role": role,
                "content": content
            })

    return messages

In [108]:
def convert_ultrafeedback(sample):
    return sample["chosen"]


def convert_magpie(sample):
    return sample["chosen"]

In [109]:
CONVERTERS = {
    "openhermes": convert_openhermes,
    "openorca": convert_openorca,
    "ultrachat": convert_ultrachat,
    "wildchat": convert_wildchat,
    "ultrafeedback": convert_ultrafeedback,
    "magpie": convert_magpie,
}

In [110]:
def is_valid_conversation(messages):
    if not messages:
        return False

    if len(messages) < 2:
        return False

    if messages[0]["role"] not in ("system", "user"):
        return False

    assistant_count = 0

    for msg in messages:
        role = msg["role"]
        content = msg["content"].strip()

        if role not in ("system", "user", "assistant"):
            return False

        if len(content) < 2:
            return False

        if len(content) > 12000:
            return False

        if role == "assistant":
            assistant_count += 1

    if assistant_count == 0:
        return False

    return True

In [111]:
ROLE_PREFIX = {
    "system": "<|system|>\n",
    "user": "<|user|>\n",
    "assistant": "<|assistant|>\n",
}

In [112]:
def conversation_to_text(messages):
    text = ""

    for msg in messages:
        text += ROLE_PREFIX[msg["role"]]
        text += msg["content"].strip()
        text += "\n\n"

    return text.strip()

In [113]:
def encode_conversation(messages):
    text = conversation_to_text(messages)
    ids = tokenizer.encode(text).ids
    return [BOS_ID] + ids + [EOS_ID]

In [114]:
DATASET_LIMITS = {
    "openhermes": None,
    "openorca": None,
    "ultrachat": None,
    "wildchat": None,
    "ultrafeedback": None,
    "magpie": None,
}

stats = {
    "train_conversations": 0,
    "val_conversations": 0,
    "train_tokens": 0,
    "val_tokens": 0,
}

In [115]:
BUFFER_SIZE = 1_000_000

train_buffer = []
val_buffer = []

In [116]:
def flush_train():
    global train_buffer, train_tokens

    if train_buffer:
        arr = np.asarray(train_buffer, dtype=np.uint16)
        arr.tofile(train_file)
        train_tokens += len(arr)
        train_buffer.clear()

In [117]:
def flush_val():
    global val_buffer, val_tokens

    if val_buffer:
        arr = np.asarray(val_buffer, dtype=np.uint16)
        arr.tofile(val_file)
        val_tokens += len(arr)
        val_buffer.clear()

In [118]:
def write_tokens(token_ids):
    if random.random() < 0.99:
        train_buffer.extend(token_ids)
        stats["train_conversations"] += 1

        if len(train_buffer) >= BUFFER_SIZE:
            flush_train()

    else:
        val_buffer.extend(token_ids)
        stats["val_conversations"] += 1

        if len(val_buffer) >= BUFFER_SIZE:
            flush_val()

In [119]:

import re
import html
import unicodedata
from bs4 import BeautifulSoup

IDENTITY_PATTERNS = [
    (re.compile(r"\bI am ChatGPT\b", re.I), "I am Virgo"),
    (re.compile(r"\bI'm ChatGPT\b", re.I), "I'm Virgo"),
    (re.compile(r"\bAs ChatGPT\b", re.I), "As Virgo"),
    (re.compile(r"\bI am Claude\b", re.I), "I am Virgo"),
    (re.compile(r"\bI'm Claude\b", re.I), "I'm Virgo"),
    (re.compile(r"\bI am Gemini\b", re.I), "I am Virgo"),
    (re.compile(r"\bI'm Gemini\b", re.I), "I'm Virgo"),
    (re.compile(r"\bI am Bard\b", re.I), "I am Virgo"),
    (re.compile(r"\bI'm Bard\b", re.I), "I'm Virgo"),
    (re.compile(r"\bI am Copilot\b", re.I), "I am Virgo"),
    (re.compile(r"\bI'm Copilot\b", re.I), "I'm Virgo"),
    (re.compile(r"\bI am Grok\b", re.I), "I am Virgo"),
    (re.compile(r"\bI'm Grok\b", re.I), "I'm Virgo"),
    (re.compile(r"\bI am DeepSeek\b", re.I), "I am Virgo"),
    (re.compile(r"\bI'm DeepSeek\b", re.I), "I'm Virgo"),
    (re.compile(r"\bI am Qwen\b", re.I), "I am Virgo"),
    (re.compile(r"\bI'm Qwen\b", re.I), "I'm Virgo"),

    (re.compile(r"trained by OpenAI", re.I), "trained as Virgo"),
    (re.compile(r"created by OpenAI", re.I), "created by Punit Kumar Kashyap"),
    (re.compile(r"developed by OpenAI", re.I), "developed by Punit Kumar Kashyap"),
    (re.compile(r"built by OpenAI", re.I), "built by Punit Kumar Kashyap"),

    (re.compile(r"created by Anthropic", re.I), "created by Punit Kumar Kashyap"),
    (re.compile(r"developed by Anthropic", re.I), "developed by Punit Kumar Kashyap"),

    (re.compile(r"created by Google", re.I), "created by Punit Kumar Kashyap"),
    (re.compile(r"developed by Google", re.I), "developed by Punit Kumar Kashyap"),

    (re.compile(r"created by Microsoft", re.I), "created by Punit Kumar Kashyap"),
    (re.compile(r"developed by Microsoft", re.I), "developed by Punit Kumar Kashyap"),
]

In [120]:
URL_RE = re.compile(r'https?://\S+')
MULTISPACE_RE = re.compile(r'[ \t]+')
MULTINEWLINE_RE = re.compile(r'\n{3,}')
REPEATED_PUNCT_RE = re.compile(r'([!?.,])\1{3,}')
INVISIBLE_RE = re.compile(r'[\u200B-\u200D\uFEFF\u2060]')
SEPARATOR_RE = re.compile(r'^[-=_*]{5,}$', re.MULTILINE)



def clean_text(text):
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = html.unescape(text)

    HTML_TAG_RE = re.compile(r"<[^>]+>")

    text = html.unescape(text)
    text = HTML_TAG_RE.sub("", text)

    text = INVISIBLE_RE.sub("", text)

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    text = SEPARATOR_RE.sub("", text)

    text = MULTISPACE_RE.sub(" ", text)

    text = MULTINEWLINE_RE.sub("\n\n", text)

    text = REPEATED_PUNCT_RE.sub(lambda m: m.group(1) * 3, text)

    text = URL_RE.sub(lambda m: m.group(0).split("?")[0], text)

    for pattern, replacement in IDENTITY_PATTERNS:
        text = pattern.sub(replacement, text)

    return text.strip()

In [121]:
VALID_ROLES = {"system", "user", "assistant"}

def clean_conversation(messages):
    cleaned = []
    previous = None

    for msg in messages:
        role = msg.get("role", "").strip().lower()
        content = clean_text(msg.get("content", ""))

        if role not in VALID_ROLES:
            continue

        if not content:
            continue

        if previous is not None:
            if previous["role"] == role:
                previous["content"] += "\n\n" + content
                continue

        current = {
            "role": role,
            "content": content
        }

        cleaned.append(current)
        previous = current

    while cleaned and cleaned[0]["role"] == "assistant":
        cleaned.pop(0)

    while cleaned and cleaned[-1]["role"] != "assistant":
        cleaned.pop()

    if len(cleaned) < 2:
        return None

    assistant = 0
    user = 0

    for msg in cleaned:
        if msg["role"] == "assistant":
            assistant += 1
        elif msg["role"] == "user":
            user += 1

    if assistant == 0 or user == 0:
        return None

    return cleaned

In [122]:
import re

URL_RE = re.compile(r'https?://')
CODE_RE = re.compile(r'```')
WORD_RE = re.compile(r'\w+')

def quality_score(messages):
    score = 100

    turns = len(messages)

    if turns < 2:
        score -= 100

    if turns > 20:
        score -= 10

    total_chars = 0
    total_words = 0
    urls = 0
    code_blocks = 0

    for msg in messages:
        text = msg["content"]

        total_chars += len(text)
        total_words += len(WORD_RE.findall(text))
        urls += len(URL_RE.findall(text))
        code_blocks += len(CODE_RE.findall(text))

    avg_words = total_words / turns

    if avg_words < 6:
        score -= 40

    if avg_words > 600:
        score -= 20

    if urls > 10:
        score -= 15

    if code_blocks > 20:
        score -= 10

    if total_chars > 50000:
        score -= 20

    return max(score, 0)

In [123]:
import hashlib

seen_conversations = set()

def conversation_hash(messages):
    text = []

    for msg in messages:
        text.append(msg["role"])
        text.append(msg["content"])

    text = "\n".join(text)

    return hashlib.sha256(text.encode("utf-8")).hexdigest()


In [124]:

def is_duplicate(messages):
    h = conversation_hash(messages)

    if h in seen_conversations:
        return True

    seen_conversations.add(h)
    return False

In [125]:
SEQ_LENGTH = 1024

train_pack = []
val_pack = []

def write_packed(token_ids):
    global train_pack, val_pack
    global train_tokens, val_tokens

    if random.random() < 0.99:

        train_pack.extend(token_ids)

        while len(train_pack) >= SEQ_LENGTH:
            chunk = np.asarray(train_pack[:SEQ_LENGTH], dtype=np.uint16)
            chunk.tofile(train_file)
            train_tokens += SEQ_LENGTH
            del train_pack[:SEQ_LENGTH]

    else:

        val_pack.extend(token_ids)

        while len(val_pack) >= SEQ_LENGTH:
            chunk = np.asarray(val_pack[:SEQ_LENGTH], dtype=np.uint16)
            chunk.tofile(val_file)
            val_tokens += SEQ_LENGTH
            del val_pack[:SEQ_LENGTH]

In [126]:
DATASET_LIMITS = {
    "openhermes": 500_000,
    "openorca": 400_000,
    "ultrachat": 400_000,
    "wildchat": 300_000,
    "ultrafeedback": 200_000,
    "magpie": 200_000,
}

In [127]:
import json
import time
import os

CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")

checkpoint = {
    "datasets": {},
    "train_tokens": train_tokens,
    "val_tokens": val_tokens,
    "time": time.time()
}

def save_checkpoint():
    checkpoint["train_tokens"] = train_tokens
    checkpoint["val_tokens"] = val_tokens
    checkpoint["time"] = time.time()

    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=4)

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return None

In [128]:
import json

with open("/kaggle/input/datasets/punitkashyap2007/virgo-identity/virgo_identity.json", "r", encoding="utf-8") as f:
    identity_dataset = json.load(f)

In [129]:
datasets["identity"] = identity_dataset

In [130]:
def convert_identity(sample):
    return sample["messages"]

CONVERTERS["identity"] = convert_identity

IDENTITY_REPEAT = 20

In [131]:
total_processed = 0
total_kept = 0
total_duplicates = 0
total_quality = 0
total_invalid = 0
total_errors = 0

In [132]:
from itertools import repeat

IDENTITY_REPEAT = 20

for dataset_name, dataset in datasets.items():

    print(f"\n{'='*80}")
    print(f"PROCESSING {dataset_name.upper()}")
    print(f"{'='*80}")

    converter = CONVERTERS[dataset_name]

    processed = 0
    kept = 0
    duplicates = 0
    quality_removed = 0
    invalid = 0
    errors = 0

    limit = DATASET_LIMITS.get(dataset_name)

    if dataset_name == "identity":
        iterator = (
            sample
            for _ in range(IDENTITY_REPEAT)
            for sample in dataset
        )
        total = len(dataset) * IDENTITY_REPEAT
    else:
        iterator = dataset
        total = limit

    pbar = tqdm(total=total, desc=dataset_name)

    for sample in iterator:

        if limit is not None and processed >= limit:
            break

        processed += 1
        total_processed += 1

        try:

            if dataset_name == "wildchat":

                language = str(sample.get("language", "")).lower()

                if language not in ("english", "en"):
                    continue

                if sample.get("toxic", False):
                    continue

            messages = converter(sample)
            messages = clean_conversation(messages)

            if messages is None:
                invalid += 1
                total_invalid += 1
                continue

            if not is_valid_conversation(messages):
                invalid += 1
                total_invalid += 1
                continue

            score = quality_score(messages)

            if score < 70:
                quality_removed += 1
                total_quality += 1
                continue

            if is_duplicate(messages):
                duplicates += 1
                total_duplicates += 1
                continue

            token_ids = encode_conversation(messages)

            if len(token_ids) < 16:
                invalid += 1
                total_invalid += 1
                continue

            write_packed(token_ids)

            kept += 1
            total_kept += 1

        except Exception as e:
            errors += 1
            total_errors += 1

            print(f"\nERROR in {dataset_name}")
            print(type(e).__name__, e)
            print(sample)
            raise

        if processed % 1000 == 0:
            pbar.set_postfix(
                kept=kept,
                dup=duplicates,
                quality=quality_removed,
                invalid=invalid,
                train_tokens=train_tokens,
                val_tokens=val_tokens
            )

        if processed % 10000 == 0:
            flush_train()
            flush_val()

            try:
                save_checkpoint()
            except:
                pass

        pbar.update(1)

    pbar.close()

    print(f"\nDataset : {dataset_name}")
    print(f"Processed : {processed:,}")
    print(f"Kept : {kept:,}")
    print(f"Duplicates : {duplicates:,}")
    print(f"Quality Removed : {quality_removed:,}")
    print(f"Invalid : {invalid:,}")
    print(f"Errors : {errors:,}")


PROCESSING OPENHERMES


openhermes:   0%|          | 0/500000 [00:00<?, ?it/s]


Dataset : openhermes
Processed : 500,000
Kept : 499,492
Duplicates : 15
Quality Removed : 79
Invalid : 414
Errors : 0

PROCESSING OPENORCA


openorca:   0%|          | 0/400000 [00:00<?, ?it/s]


Dataset : openorca
Processed : 400,000
Kept : 396,052
Duplicates : 574
Quality Removed : 4
Invalid : 3,370
Errors : 0

PROCESSING ULTRACHAT


...


Dataset : ultrafeedback
Processed : 60,917
Kept : 60,503
Duplicates : 72
Quality Removed : 19
Invalid : 323
Errors : 0

PROCESSING MAGPIE


magpie:   0%|          | 0/200000 [00:00<?, ?it/s]


Dataset : magpie
Processed : 98,000
Kept : 97,999
Duplicates : 1
Quality Removed : 0
Invalid : 0
Errors : 0

PROCESSING IDENTITY


identity:   0%|          | 0/2000 [00:00<?, ?it/s]


Dataset : identity
Processed : 2,000
Kept : 89
Duplicates : 1,691
Quality Removed : 220
Invalid : 0
Errors : 0


In [133]:
for _ in range(20):
    for sample in identity_dataset:
        messages = convert_identity(sample)
        messages = clean_conversation(messages)

        if messages is None:
            continue

        token_ids = encode_conversation(messages)
        write_packed(token_ids)

In [134]:
flush_train()
flush_val()

if train_pack:
    train_pack.extend([EOS_ID] * (SEQ_LENGTH - len(train_pack)))
    np.asarray(train_pack, dtype=np.uint16).tofile(train_file)

train_file.close()
val_file.close()

In [135]:
import os

print("Train Tokens:", os.path.getsize(TRAIN_BIN) // 2)
print("Validation Tokens:", os.path.getsize(VAL_BIN) // 2)

Train Tokens: 889659392
Validation Tokens: 8868864


In [136]:
import numpy as np
import random

train = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")

start = random.randint(0, len(train) - 512)

print(tokenizer.decode(train[start:start+512].tolist()))

 outcome. A skilled player may be unable to carry a team with weaker players or poor coordination.5. **Psychological factors**: Fatigue, tilt, and mental state can all affect a player's performance. A skilled player may underperform due to mental exhaustion or frustration.6. **Hardware and software limitations**: The quality of a player's hardware, such as their computer or console, can influence their performance. For example, a player with a lower-end graphics card may struggle to maintain a high frame rate in a demanding game.7. **Game knowledge and strategy**: Familiarity with a game's mechanics, maps, and strategies can give players an advantage. A skilled player may still lose to a less skilled opponent who has a deeper understanding of the game's intricacies.8. **Chance and luck**: Even in games with minimal RNG involvement, chance events can occur. For example, a misplaced shot or an accidental misstep can lead to an unfortunate outcome.9. **Social dynamics**: In games with a s

In [137]:
text = tokenizer.decode(train[:2_000_000].tolist())

keywords = [
    "ChatGPT",
    "OpenAI",
    "Claude",
    "Gemini",
    "Anthropic",
    "Google AI"
]

for k in keywords:
    print(k, text.count(k))

ChatGPT 0
OpenAI 0
Claude 83
Gemini 12
Anthropic 0
Google AI 0


In [138]:
keywords = [
    "I am Claude",
    "I'm Claude",
    "I am Gemini",
    "I'm Gemini",
    "Created by Anthropic",
    "Developed by Anthropic",
    "Created by Google",
    "Developed by Google",
]

for k in keywords:
    print(k, text.count(k))

I am Claude 0
I'm Claude 0
I am Gemini 0
I'm Gemini 0
Created by Anthropic 0
Developed by Anthropic 0
Created by Google 0
Developed by Google 0


In [139]:
import os
import zipfile

OUTPUT_DIR = "/kaggle/working/virgo_chat"

FILES = [
    "train.bin",
    "val.bin",
    "virgo_tokenizer.json",
    "identity.json",
    "checkpoint.json"
]

zip_path = "/kaggle/working/virgo_chat_dataset.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    for file in FILES:
        path = os.path.join(OUTPUT_DIR, file)
        if os.path.exists(path):
            zf.write(path, arcname=file)
            print(f"Added: {file}")
        else:
            print(f"Skipped: {file}")

print("\nZIP created successfully!")
print(zip_path)

size = os.path.getsize(zip_path) / (1024**3)
print(f"Size: {size:.2f} GB")

Added: train.bin
Added: val.bin
Skipped: virgo_tokenizer.json
Skipped: identity.json
Added: checkpoint.json

ZIP created successfully!
/kaggle/working/virgo_chat_dataset.zip
Size: 0.87 GB


In [142]:
import random
import numpy as np
import re

train = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")

for i in range(10):

    idx = random.randint(0, len(train) - 1)

    left = idx
    while left > 0 and train[left] != EOS_ID:
        left -= 1

    right = idx
    while right < len(train) - 1 and train[right] != EOS_ID:
        right += 1

    tokens = train[left + 1:right].tolist()
    text = tokenizer.decode(tokens)

    text = re.sub(r"<\|system\|>", "\n🟢 SYSTEM\n" + "-" * 80 + "\n", text)
    text = re.sub(r"<\|user\|>", "\n🔵 USER\n" + "-" * 80 + "\n", text)
    text = re.sub(r"<\|assistant\|>", "\n🟣 ASSISTANT\n" + "-" * 80 + "\n", text)

    print("\n" + "=" * 120)
    print(f"CONVERSATION {i + 1}")
    print("=" * 120)
    print(text.strip())
    print("=" * 120 + "\n")


CONVERSATION 1
🔵 USER
--------------------------------------------------------------------------------
In order to perform accurate calculations for complex mathematical equations with multiple operations using Java code, can you please provide a detailed algorithm or approach that covers various scenarios? For instance, how would your code handle equations involving variables, exponentiation, trigonometric functions or logarithmic functions? Additionally, could you provide some sample Java code that demonstrates how to solve an equation that involves (4 * 5) / (7 - 2) + (10 / 2) * 3, ensuring that the answer is precise and can handle decimal values? Thank you.
🟣 ASSISTANT
--------------------------------------------------------------------------------
Sure, I'd be happy to help you with that!To create a Java algorithm or approach for solving mathematical equations with multiple operations, you can follow these steps:1. Identify the operations involved in the equation. For example, in

In [149]:
import numpy as np

train = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")

train_tokens = len(train)
val_tokens = len(val)
total_tokens = train_tokens + val_tokens

print("=" * 70)
print("VIRGO CHAT DATASET")
print("=" * 70)
print(f"Train Tokens : {train_tokens:,}")
print(f"Val Tokens   : {val_tokens:,}")
print(f"Total Tokens : {round(total_tokens/1_000_000, 2)}M")
print("=" * 70)

VIRGO CHAT DATASET
Train Tokens : 889,659,392
Val Tokens   : 8,868,864
Total Tokens : 898.53M
